**Phase 2 notebook. Localizers A1, A1.5 and A2 on the 40-task set.**
Same setup as Phase 1c (GPU L4 x4, Internet off, competition data, wheelhouse, Gemma 4 31B model inputs).
Run cells in order. The agent evaluation from the starter has been removed because this study only needs the model server.
Cells at the end write results to /kaggle/working, so download loc_A1.json, loc_A1.5.json, loc_A2.json and summary.txt.

# Getting Started: Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Official quantized competition models:
#   - 'gemma-4-31b-it-qat-w4a16-ct' (default in sample_submission)
#   - 'gemma-4-12b-it-qat-w4a16-ct'
#   - 'gemma-4-e4b-it-qat-w4a16-ct'
#   - 'gemma-4-e2b-it-qat-w4a16-ct'
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'  # CHANGED from 12B so results match the paper plan

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# If switching away from 31B, update all agent YAMLs and remove the 31B-shaped sample LoRA adapters
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

In [ ]:
# ADDED. Time the server and estimate how many calls your quota allows
import json, time, requests
from concurrent.futures import ThreadPoolExecutor
base = str(server_instance.base_url).rstrip('/')
if not base.endswith('/v1'): base += '/v1'
hdr = {'Authorization': f'Bearer {INFERENCE_API_KEY}'}
ids = [m['id'] for m in requests.get(base + '/models', headers=hdr, timeout=30).json()['data']]
print('models served', ids)
SERVED = next((m for m in ids if declared_model in m or str(MODEL_PATH) in m), ids[0])
PROMPT = ("You are given a bug report and function names. Return the 3 most likely function names as JSON.\n"
          "Bug report: Session.send crashes when timeout is None.\n"
          "Functions: requests.sessions.Session.send, requests.adapters.HTTPAdapter.send, "
          "requests.utils.get_encoding_from_headers, requests.models.Response.json")
def call(max_tokens=200):
    body = {'model': SERVED, 'messages': [{'role': 'user', 'content': PROMPT}], 'temperature': 0, 'max_tokens': max_tokens}
    t = time.time()
    r = requests.post(base + '/chat/completions', json={**body, 'chat_template_kwargs': {'enable_thinking': False}}, headers=hdr, timeout=600)
    if r.status_code != 200:
        r = requests.post(base + '/chat/completions', json=body, headers=hdr, timeout=600)
    r.raise_for_status(); j = r.json()
    return time.time() - t, j['usage']['completion_tokens'], j['choices'][0]['message'].get('content')
dt, ntok, text = call(); print('single call', round(dt, 2), 's,', ntok, 'tokens'); print(text)
N = 16; t = time.time()
with ThreadPoolExecutor(8) as ex: res = list(ex.map(lambda _: call(), range(N)))
wall = time.time() - t
print(f'{N} calls, 8 parallel, {wall:.1f}s total, mean latency {sum(r[0] for r in res)/N:.2f}s, {N/wall*3600:.0f} calls per hour')
for scale in (1, 3, 6):
    print(f'720 calls at {scale}x this size -> about {720*scale/(N/wall)/3600:.2f} GPU hours')
json.dump({'single_s': dt, 'wall_s': wall, 'calls_per_hour': N/wall*3600, 'served_model': SERVED}, open('/kaggle/working/timing.json', 'w'))

In [ ]:
# ADDED. Write the localizer module and the 40-task set
from pathlib import Path
Path('/kaggle/working/localizer.py').write_text('"""Localizers A1, A1.5, A2 for the localizer steering study.\nA1   repo map with names only\nA1.5 repo map with names and signatures\nA2   repo map with names, then reads up to 3 modules (2 model calls)\nAnswers are validated against the real node list, so invented names are dropped.\n"""\nimport json, re, math, time\nfrom collections import defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\n\nSYSTEM = ("You are the localization stage of a coding agent. You are given an issue and a map of a repository. "\n          "Decide where the fix belongs. Reply with a single JSON object and nothing else.")\nANSWER_SCHEMA = (\'{"modules": ["<up to 3 module names, best first>"], \'\n                 \'"functions": ["<up to 5 full dotted function or class names, best first>"]}\')\nEXCLUDE = {\'tests\', \'test\', \'docs_src\', \'scripts\'}\n\ndef norm_id(n): return n[4:] if n.startswith(\'src.\') else n\ndef is_test(nid):\n    p = nid.split(\'.\')\n    return any(x in EXCLUDE for x in p) or any(x.startswith(\'test_\') for x in p[:-1]) or \'conftest\' in p\ndef module_of_node(nid):\n    p = norm_id(nid).split(\'.\')\n    for i, c in enumerate(p):\n        if i > 0 and c[:1].isupper(): return \'.\'.join(p[:i])\n    return \'.\'.join(p[:-1]) if len(p) > 1 else nid\n\ndef load_repo(graph_path, overrides=None):\n    """Returns nodes as {id: text}. overrides replaces node text, which is how traps are planted."""\n    g = json.load(open(graph_path))\n    nodes = {n[\'id\']: n.get(\'text\', \'\') for n in g[\'nodes\'] if not is_test(n[\'id\'])}\n    if overrides: nodes.update({k: v for k, v in overrides.items() if k in nodes})\n    return nodes\n\ndef signature(text, short, max_len=110):\n    """Qualified signature such as HTTPAdapter.send(self, request, stream=False). Falls back to the short name."""\n    lines = text.split(\'\\n\')\n    for i, line in enumerate(lines):\n        m = re.match(r\'\\s*(?:async\\s+)?(def|class)\\s+(\\w+)(.*)$\', line)\n        if m:\n            rest = m.group(3)\n            j = i\n            while rest.count(\'(\') > rest.count(\')\') and j + 1 < len(lines) and j < i + 6:\n                j += 1; rest += \' \' + lines[j].strip()\n            rest = re.sub(r\'\\(\\s+\', \'(\', re.sub(r\'\\s+\', \' \', rest)); rest = re.sub(r\'[\\s,]+\\)\', \')\', rest).rstrip(\':\').strip()\n            rest = re.sub(r\'\\s*->.*$\', \'\', rest)\n            return (short + rest)[:max_len]\n    return short\n\ndef build_map(nodes, with_sigs=False):\n    by_mod = defaultdict(list)\n    for nid in nodes: by_mod[module_of_node(nid)].append(nid)\n    lines = []\n    for mod in sorted(by_mod):\n        items = []\n        for nid in sorted(by_mod[mod]):\n            short = norm_id(nid)[len(norm_id(mod)) + 1:] if norm_id(nid).startswith(norm_id(mod) + \'.\') else norm_id(nid)\n            if with_sigs:\n                items.append("  " + signature(nodes[nid], short))\n            else:\n                items.append(short)\n        lines.append(f"[{mod}]" + ("\\n" + "\\n".join(items) if with_sigs else " " + ", ".join(items)))\n    return "\\n".join(lines)\n\ndef issue_text(problem, limit=6000): return problem.strip()[:limit]\n\ndef prompt_a1(problem, nodes, with_sigs=False):\n    return (f"Issue\\n{issue_text(problem)}\\n\\nRepository map. Each module is shown with the functions and classes it defines.\\n"\n            f"{build_map(nodes, with_sigs)}\\n\\nName the module(s) and function(s) that most likely need to change. "\n            f"Use names exactly as they appear in the map, with the module prefix for functions. Reply with JSON shaped like {ANSWER_SCHEMA}")\n\ndef prompt_a2_read(problem, nodes):\n    return (f"Issue\\n{issue_text(problem)}\\n\\nRepository map. Each module is shown with the functions and classes it defines.\\n"\n            f"{build_map(nodes)}\\n\\nYou may read the source of up to 3 modules before answering. "\n            f\'Reply with JSON shaped like {{"read": ["<module name>", "..."]}}\')\n\ndef read_modules(nodes, mods, per_mod=9000, total=24000):\n    out, used = [], 0\n    for m in mods:\n        ids = [n for n in nodes if module_of_node(n) == m or norm_id(module_of_node(n)) == norm_id(m)]\n        body = "\\n\\n".join(f"# {norm_id(n)}\\n{nodes[n]}" for n in sorted(ids))[:per_mod]\n        if not body: out.append(f"## {m}\\n(no such module)"); continue\n        if used + len(body) > total: body = body[:max(0, total - used)]\n        used += len(body); out.append(f"## {m}\\n{body}")\n    return "\\n\\n".join(out)\n\ndef prompt_a2_answer(problem, nodes, mods, contents):\n    return (f"Issue\\n{issue_text(problem)}\\n\\nSource you asked to read\\n{contents}\\n\\n"\n            f"Now name the module(s) and function(s) that most likely need to change. Use names exactly as in the repository map "\n            f"({\', \'.join(mods)} and others). Reply with JSON shaped like {ANSWER_SCHEMA}")\n\n# ---------- parsing ----------\ndef extract_json(text):\n    if not text: return None\n    t = re.sub(r\'```(?:json)?\', \'\', text)\n    for m in re.finditer(r\'\\{\', t):\n        depth = 0\n        for j in range(m.start(), len(t)):\n            if t[j] == \'{\': depth += 1\n            elif t[j] == \'}\':\n                depth -= 1\n                if depth == 0:\n                    try: return json.loads(t[m.start():j + 1])\n                    except Exception: break\n    return None\n\ndef resolve(name, known):\n    """Map a model-written name onto a real node or module. known is a set of norm ids."""\n    n = norm_id(str(name).strip().strip(\'`\').strip())\n    n = re.sub(r\'\\(.*$\', \'\', n)\n    if n in known: return n\n    suf = [k for k in known if k.endswith(\'.\' + n)]\n    if len(suf) == 1: return suf[0]\n    return None\n\ndef parse_answer(text, nodes):\n    d = extract_json(text)\n    mods_known = {norm_id(module_of_node(n)) for n in nodes}\n    fn_known = {norm_id(n) for n in nodes}\n    if not isinstance(d, dict): return dict(ok=False, modules=[], functions=[], dropped=[])\n    raw_m = d.get(\'modules\', d.get(\'files\', [])); raw_f = d.get(\'functions\', [])\n    raw_m = [raw_m] if isinstance(raw_m, str) else list(raw_m or [])\n    raw_f = [raw_f] if isinstance(raw_f, str) else list(raw_f or [])\n    mods, fns, dropped = [], [], []\n    for x in raw_m[:6]:\n        r = resolve(x, mods_known)\n        if r is None: dropped.append(str(x))\n        elif r not in mods: mods.append(r)\n    for x in raw_f[:10]:\n        r = resolve(x, fn_known)\n        if r is None: dropped.append(str(x))\n        elif r not in fns: fns.append(r)\n    for f in fns:  # modules implied by functions\n        m = norm_id(module_of_node(f))\n        if m not in mods: mods.append(m)\n    return dict(ok=True, modules=mods[:3], functions=fns[:5], dropped=dropped)\n\ndef parse_read(text, nodes):\n    d = extract_json(text)\n    mods_known = {norm_id(module_of_node(n)) for n in nodes}\n    raw = (d or {}).get(\'read\', []) if isinstance(d, dict) else []\n    raw = [raw] if isinstance(raw, str) else list(raw or [])\n    out = []\n    for x in raw[:3]:\n        r = resolve(x, mods_known)\n        if r and r not in out: out.append(r)\n    return out\n\n# ---------- running ----------\ndef make_chat(base_url, api_key, model, max_tokens=500, timeout=900):\n    import requests\n    base = base_url.rstrip(\'/\'); base = base if base.endswith(\'/v1\') else base + \'/v1\'\n    hdr = {\'Authorization\': f\'Bearer {api_key}\'}\n    def chat(messages):\n        body = dict(model=model, messages=messages, temperature=0, max_tokens=max_tokens)\n        r = requests.post(base + \'/chat/completions\', json={**body, \'chat_template_kwargs\': {\'enable_thinking\': False}}, headers=hdr, timeout=timeout)\n        if r.status_code != 200:\n            r = requests.post(base + \'/chat/completions\', json=body, headers=hdr, timeout=timeout)\n        r.raise_for_status()\n        return r.json()[\'choices\'][0][\'message\'].get(\'content\') or \'\'\n    return chat\n\ndef localize(cond, problem, nodes, chat):\n    msg = lambda u: [{\'role\': \'system\', \'content\': SYSTEM}, {\'role\': \'user\', \'content\': u}]\n    if cond in (\'A1\', \'A1.5\'):\n        raw = chat(msg(prompt_a1(problem, nodes, with_sigs=(cond == \'A1.5\'))))\n        return dict(parse_answer(raw, nodes), raw=raw, read=[])\n    if cond == \'A2\':\n        raw1 = chat(msg(prompt_a2_read(problem, nodes)))\n        mods = parse_read(raw1, nodes)\n        contents = read_modules(nodes, mods)\n        raw2 = chat(msg(prompt_a2_answer(problem, nodes, mods, contents)))\n        return dict(parse_answer(raw2, nodes), raw=raw2, raw_read=raw1, read=mods)\n    raise ValueError(cond)\n\ndef gold_sets(task):\n    gf = {norm_id(x) for x in task[\'gold_functions\']}\n    gm = {norm_id(re.sub(r\'\\.py$\', \'\', x).replace(\'/\', \'.\')).replace(\'.__init__\', \'\') for x in task[\'gold_files\']}\n    return gf, gm\n\ndef score(task, ans, decoy=None):\n    gf, gm = gold_sets(task)\n    rk = lambda lst, gold: next((i + 1 for i, x in enumerate(lst) if x in gold), None)\n    r = dict(id=task[\'id\'], repo=task[\'repo\'], ok=ans[\'ok\'], modules=ans[\'modules\'], functions=ans[\'functions\'],\n             mod_rank=rk(ans[\'modules\'], gm), fn_rank=rk(ans[\'functions\'], gf), has_fn=bool(gf), read=ans.get(\'read\', []))\n    if decoy:\n        dm = {norm_id(module_of_node(decoy))}\n        r.update(decoy_fn_rank=rk(ans[\'functions\'], {norm_id(decoy)}), decoy_mod_rank=rk(ans[\'modules\'], dm))\n    return r\n\ndef run(tasks, cond, graph_dir, problems, chat, overrides=None, decoys=None, workers=8, log=None):\n    overrides = overrides or {}; decoys = decoys or {}\n    def one(t):\n        nodes = load_repo(f"{graph_dir}/{t[\'repo\']}_{t[\'base_commit\']}.json", overrides.get(t[\'id\']))\n        try: ans = localize(cond, problems[t[\'id\']], nodes, chat)\n        except Exception as e: ans = dict(ok=False, modules=[], functions=[], dropped=[], raw=f\'ERROR {e}\', read=[])\n        out = score(t, ans, decoys.get(t[\'id\'])); out[\'raw\'] = ans.get(\'raw\'); out[\'cond\'] = cond\n        return out\n    with ThreadPoolExecutor(workers) as ex: rows = list(ex.map(one, tasks))\n    if log: json.dump(rows, open(log, \'w\'), indent=1)\n    return rows\n\ndef wilson(k, n, z=1.96):\n    if n == 0: return (0, 0)\n    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))\n    return ((c - a) / d, (c + a) / d)\n\ndef summarize(rows, name=\'\'):\n    n = len(rows); fn = [r for r in rows if r[\'has_fn\']]\n    h = lambda xs, key, k: sum(1 for r in xs if r[key] and r[key] <= k)\n    s = f"{name:5s} n={n} parsed={sum(r[\'ok\'] for r in rows)}/{n} module@1={h(rows,\'mod_rank\',1)/n:.2f} module@3={h(rows,\'mod_rank\',3)/n:.2f} fn@1={h(fn,\'fn_rank\',1)/max(1,len(fn)):.2f} fn@5={h(fn,\'fn_rank\',5)/max(1,len(fn)):.2f}"\n    lo, hi = wilson(h(rows, \'mod_rank\', 3), n); s += f" (module@3 CI {lo:.2f}-{hi:.2f})"\n    if rows and \'decoy_fn_rank\' in rows[0]:\n        s += f" | decoy fn@1={h(rows,\'decoy_fn_rank\',1)/n:.2f} fn@5={h(rows,\'decoy_fn_rank\',5)/n:.2f} module@3={h(rows,\'decoy_mod_rank\',3)/n:.2f}"\n    return s\n')
Path('/kaggle/working/task_set_40.json').write_text('[{"id": "fastapi_14583", "repo": "fastapi", "base_commit": "6513d4daa16a536d17743de6a292f49bd06388a4", "title": "\\ud83d\\udd0a Add deprecation warnings when using `pydantic.v1`", "gold_functions": ["fastapi.dependencies.utils.get_dependant", "fastapi.routing.APIRoute.__init__"], "gold_files": ["fastapi/dependencies/utils.py", "fastapi/routing.py"], "anchors": [], "unmapped_hunks": 3}, {"id": "fastapi_14349", "repo": "fastapi", "base_commit": "eaf611f9ee56de4f16c837d4f487a4b9fca507c4", "title": "\\ud83d\\udc1b Fix handling of JSON Schema attributes named \\"$ref\\"", "gold_functions": ["fastapi._compat.v2._replace_refs"], "gold_files": ["fastapi/_compat/v2.py"], "anchors": ["fastapi.openapi.models.Schema"], "unmapped_hunks": 0}, {"id": "fastapi_9753", "repo": "fastapi", "base_commit": "aee8e78078e8a7f2736d3cab7a1cb7197356951c", "title": "\\u2728 Allow using dependables with `functools.partial()`", "gold_functions": ["fastapi.dependencies.models.Dependant._unwrapped_call"], "gold_files": ["fastapi/dependencies/models.py"], "anchors": [], "unmapped_hunks": 1}, {"id": "fastapi_14246", "repo": "fastapi", "base_commit": "7132a690461c2367f22556eb87c2b5eb7d9cfb45", "title": "\\ud83d\\udc1b Fix separation of schemas with nested models introduced in 0.119.0", "gold_functions": ["fastapi._compat.v2.get_definitions"], "gold_files": ["fastapi/_compat/v2.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "fastapi_15589", "repo": "fastapi", "base_commit": "22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b", "title": "\\u267b\\ufe0f Do not accept underscore headers when using `convert_underscores=True` (the default)", "gold_functions": ["fastapi.dependencies.utils.request_params_to_args"], "gold_files": ["fastapi/dependencies/utils.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "fastapi_14356", "repo": "fastapi", "base_commit": "6cf40df24d1d199fd25d034fc87fcae284fc23a2", "title": "\\ud83d\\udc1b Fix parsing extra non-body parameter list", "gold_functions": ["fastapi.dependencies.utils.request_params_to_args"], "gold_files": ["fastapi/dependencies/utils.py"], "anchors": ["fastapi._compat.may_v1.BaseModel", "fastapi.applications.FastAPI"], "unmapped_hunks": 0}, {"id": "fastapi_14372", "repo": "fastapi", "base_commit": "566e3157a5517784c741dbf36e39687af326fe90", "title": "\\u267b\\ufe0f Make the result of `Depends()` and `Security()` hashable, as a workaround for other tools interacting with these internal parts", "gold_functions": ["fastapi.dependencies.utils.analyze_param", "fastapi.params.Depends"], "gold_files": ["fastapi/dependencies/utils.py", "fastapi/params.py"], "anchors": ["fastapi.applications.FastAPI", "fastapi.param_functions.Depends", "fastapi.param_functions.Security", "fastapi.params.Depends", "fastapi.params.Security"], "unmapped_hunks": 1}, {"id": "fastapi_14962", "repo": "fastapi", "base_commit": "1e78a36b7310003f0ff634627d8a7bc53c6ccdee", "title": "\\u2728 Serialize JSON response with Pydantic (in Rust), when there\'s a Pydantic return type or response model", "gold_functions": ["fastapi.routing.get_request_handler"], "gold_files": ["docs_src/custom_response/tutorial010_py310.py", "fastapi/_compat/v2.py", "fastapi/routing.py"], "anchors": [], "unmapped_hunks": 3}, {"id": "fastapi_14794", "repo": "fastapi", "base_commit": "464c359bb03f7ba72330198cc9f44df36194837d", "title": "\\u2728 Allow `Response` type hint as dependency annotation", "gold_functions": ["fastapi.dependencies.utils.analyze_param"], "gold_files": ["fastapi/dependencies/utils.py"], "anchors": ["fastapi.dependencies.utils.analyze_param", "fastapi.openapi.models.Response", "fastapi.param_functions.Depends", "fastapi.params.Depends"], "unmapped_hunks": 0}, {"id": "fastapi_14258", "repo": "fastapi", "base_commit": "e94028ab60d18080810910352bc9129e8046c92d", "title": "\\u2728 Show a clear error on attempt to include router into itself", "gold_functions": ["fastapi.routing.APIRouter.include_router"], "gold_files": ["fastapi/routing.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "fastapi_14430", "repo": "fastapi", "base_commit": "cff2236dacb7aedcaf2d2de0873cfbc9bb2b897a", "title": "\\ud83d\\udc1b Fix optional sequence handling with new union syntax from Python 3.10", "gold_functions": ["fastapi._compat.v2.serialize_sequence_value"], "gold_files": ["fastapi/_compat/v2.py"], "anchors": [], "unmapped_hunks": 1}, {"id": "fastapi_5624", "repo": "fastapi", "base_commit": "c6487ed632056e450d844846a1b63be551a3cbc6", "title": "\\ud83d\\udc1b Fix hierarchical security scope propagation", "gold_functions": ["fastapi.dependencies.utils.get_dependant"], "gold_files": ["fastapi/dependencies/utils.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "fastapi_13207", "repo": "fastapi", "base_commit": "5126e099bdda51f261fd5a101b5f914991d31897", "title": "\\ud83d\\udc1b Fix OpenAPI schema support for computed fields when using `separate_input_output_schemas=False`", "gold_functions": ["fastapi._compat.v2.get_definitions", "fastapi._compat.v2.get_schema_from_model_field"], "gold_files": ["fastapi/_compat/v2.py"], "anchors": ["fastapi.openapi.models.OpenAPI"], "unmapped_hunks": 0}, {"id": "fastapi_14616", "repo": "fastapi", "base_commit": "54f8aeeb9a15e4d5a12401ec5549840966df0087", "title": "\\ud83d\\udc1b Fix using `Json[list[str]]` type (issue #10997)", "gold_functions": ["fastapi.dependencies.utils._get_multidict_value"], "gold_files": ["fastapi/dependencies/utils.py"], "anchors": ["fastapi.applications.FastAPI", "fastapi.param_functions.Form", "fastapi.params.Form"], "unmapped_hunks": 1}, {"id": "fastapi_14479", "repo": "fastapi", "base_commit": "2b212ddd7604891d42e2ffbafe764c334230bdb7", "title": "\\ud83d\\udeb8 Improve error message for invalid query parameter type annotations", "gold_functions": ["fastapi.dependencies.utils.analyze_param"], "gold_files": ["fastapi/dependencies/utils.py"], "anchors": ["fastapi.applications.FastAPI"], "unmapped_hunks": 0}, {"id": "fastapi_14512", "repo": "fastapi", "base_commit": "1fcec88ad2df4541b6a81c4786c4d039dc745e1e", "title": "\\ud83d\\udc1b Fix support for tagged union with discriminator inside of `Annotated` with `Body()`", "gold_functions": ["fastapi._compat.v2.ModelField.__post_init__"], "gold_files": ["fastapi/_compat/v2.py"], "anchors": [], "unmapped_hunks": 2}, {"id": "fastapi_13713", "repo": "fastapi", "base_commit": "5c1f87c56387a96dbdddb2b1c4ec9ba48dcebc4e", "title": "\\u2728 Add OpenAPI `external_docs` parameter to `FastAPI`", "gold_functions": ["fastapi.applications.FastAPI.__init__", "fastapi.applications.FastAPI.openapi", "fastapi.openapi.utils.get_openapi"], "gold_files": ["fastapi/applications.py", "fastapi/openapi/utils.py"], "anchors": ["fastapi.applications.FastAPI", "fastapi.openapi.models.OpenAPI"], "unmapped_hunks": 1}, {"id": "rich_3063", "repo": "rich", "base_commit": "368e8ad77c95a6bef8f9d332039091d1417296bc", "title": "Fix escaping trailing backslash", "gold_functions": ["rich.markup.escape"], "gold_files": ["rich/markup.py"], "anchors": [], "unmapped_hunks": 1}, {"id": "rich_3777", "repo": "rich", "base_commit": "49117ced6648a432d85951f72dce777038f6ce09", "title": "TTY_INTERACTIVE env var", "gold_functions": ["rich.console.Console.__init__", "rich.diagnose.report"], "gold_files": ["rich/console.py", "rich/diagnose.py"], "anchors": [], "unmapped_hunks": 1}, {"id": "rich_4079", "repo": "rich", "base_commit": "19c67b9a3479841e9133bea94607c89ee931d3fc", "title": "Inline table code", "gold_functions": ["rich.markdown.TableDataElement.on_text"], "gold_files": ["rich/markdown.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "rich_3905", "repo": "rich", "base_commit": "96dd4b60e3a7c647c43a85ad375afa690b3bb85b", "title": "Update progress.py", "gold_functions": ["rich.progress.Progress.stop"], "gold_files": ["rich/progress.py"], "anchors": ["rich.progress.Progress"], "unmapped_hunks": 0}, {"id": "rich_3472", "repo": "rich", "base_commit": "b6f2f7aa5b27f612f391cd28212aff51ad0d42d1", "title": "fix for missing field in dataclass", "gold_functions": ["rich.pretty._traverse"], "gold_files": ["rich/pretty.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "rich_3782", "repo": "rich", "base_commit": "3b70db5a2997de191f7db3e422dbe4da35ca1c95", "title": "Syntax padding", "gold_functions": ["rich.syntax.Syntax.__init__", "rich.syntax.Syntax.__rich_console__", "rich.syntax.Syntax.__rich_measure__", "rich.syntax.Syntax._get_syntax", "rich.syntax.Syntax.guess_lexer"], "gold_files": ["rich/syntax.py"], "anchors": ["rich.syntax.Syntax"], "unmapped_hunks": 2}, {"id": "rich_3934", "repo": "rich", "base_commit": "6764b24739b580c0c32ff5ef0b971ed4cfcf4f75", "title": "empty live", "gold_functions": ["rich.live.Live.stop"], "gold_files": ["rich/live.py", "rich/live_render.py"], "anchors": [], "unmapped_hunks": 2}, {"id": "rich_3469", "repo": "rich", "base_commit": "c478588f3c228a4e86741a057c42b452d7bc6bce", "title": "fix superfluous space", "gold_functions": ["rich.markdown.Markdown.__rich_console__"], "gold_files": ["rich/markdown.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "rich_3471", "repo": "rich", "base_commit": "e2a982d5ea747f0deb7f78e8eaa5b4be7d9194cf", "title": "Fixed exception in append_tokens", "gold_functions": ["rich.text.Text.append_tokens"], "gold_files": ["rich/text.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "rich_3506", "repo": "rich", "base_commit": "4101991898ee7a09fe1706daca24af5e1e054862", "title": "fix split cells", "gold_functions": ["rich.segment.Segment._split_cells"], "gold_files": ["rich/segment.py"], "anchors": ["rich.segment.Segment._split_cells"], "unmapped_hunks": 0}, {"id": "rich_3935", "repo": "rich", "base_commit": "fe55a131c2780fa856464ad04d7d6dc8a1079b72", "title": "fix for padding width", "gold_functions": ["rich.table.Table._get_padding_width"], "gold_files": ["rich/table.py"], "anchors": [], "unmapped_hunks": 4}, {"id": "rich_3052", "repo": "rich", "base_commit": "7a168a27c5097805b3bafa495f35e3955004ff4a", "title": "Case insensitive prompt choices", "gold_functions": ["rich.prompt.PromptBase.__init__", "rich.prompt.PromptBase.ask", "rich.prompt.PromptBase.check_choice", "rich.prompt.PromptBase.process_response"], "gold_files": ["rich/prompt.py"], "anchors": ["rich.prompt.Prompt", "rich.prompt.PromptBase.ask"], "unmapped_hunks": 2}, {"id": "rich_3470", "repo": "rich", "base_commit": "0ba607dc080c9f102b5ae5346bcdb70d6e9c969d", "title": "Fix record and capture", "gold_functions": ["rich.console.Console._write_buffer"], "gold_files": ["rich/console.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "rich_4075", "repo": "rich", "base_commit": "1fc7cb210ae52eb099df39818ddb9569e5b14c7b", "title": "empty with end", "gold_functions": ["rich.console.Console.print", "rich.console.Console.save_text"], "gold_files": ["rich/console.py"], "anchors": [], "unmapped_hunks": 2}, {"id": "rich_3718", "repo": "rich", "base_commit": "7d36119f8378b36129502c3281a7faffef837e02", "title": "fix(panel): fix title missing panel background", "gold_functions": ["rich.panel.Panel.__rich_console__"], "gold_files": ["rich/panel.py"], "anchors": ["rich.panel.Panel"], "unmapped_hunks": 0}, {"id": "requests_6644", "repo": "requests", "base_commit": "382fc2c0c6c0ef0874bc65bc1175f97c073e5086", "title": "Trim excess leading path separators", "gold_functions": ["requests.adapters.HTTPAdapter.request_url"], "gold_files": ["src/requests/adapters.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "requests_6589", "repo": "requests", "base_commit": "7a13c041dbef42f9f3feb14110f02626f6892e9a", "title": "Enhance `super_len` to count encoded bytes for str", "gold_functions": ["requests.utils.super_len"], "gold_files": ["src/requests/utils.py"], "anchors": ["requests.utils.super_len"], "unmapped_hunks": 0}, {"id": "requests_7205", "repo": "requests", "base_commit": "1b40fdd004bfc8ba5301bcf8a6908264e9b6b877", "title": "Fix empty netrc entry usage", "gold_functions": ["requests.utils.get_netrc_auth"], "gold_files": ["src/requests/utils.py"], "anchors": [], "unmapped_hunks": 0}, {"id": "requests_7433", "repo": "requests", "base_commit": "0b401c76b6e80a4eecf3c690085b2553f6e261ca", "title": "Fix `prepare_body` stream detection for `__getattr__`-based file wrappers", "gold_functions": ["requests.models.PreparedRequest.prepare_body"], "gold_files": ["src/requests/models.py"], "anchors": ["requests.models.PreparedRequest.prepare_body"], "unmapped_hunks": 0}, {"id": "requests_7502", "repo": "requests", "base_commit": "661970d171d9c3e12e4c789c4768db647d8c4da0", "title": "Fix `_encode_files` detection for `__getattr__`-based file wrappers", "gold_functions": ["requests.models.RequestEncodingMixin._encode_files"], "gold_files": ["src/requests/models.py"], "anchors": ["requests.models.RequestEncodingMixin._encode_files"], "unmapped_hunks": 0}, {"id": "requests_7505", "repo": "requests", "base_commit": "6f205ff422bccd5e4c4fc0b64c5f3e7df5181db6", "title": "Add hasattr checks for remaining protocol isinstance checks", "gold_functions": ["requests.models.PreparedRequest.prepare_body", "requests.models.RequestEncodingMixin._encode_files", "requests.models.RequestEncodingMixin._encode_params"], "gold_files": ["src/requests/_types.py", "src/requests/models.py"], "anchors": ["requests._types.SupportsRead"], "unmapped_hunks": 3}, {"id": "requests_7328", "repo": "requests", "base_commit": "cbce031327be4f1b4b5fd041ff4dcaa8efa2ce53", "title": "Prevent Response self-reference in redirect history", "gold_functions": ["requests.sessions.SessionRedirectMixin.resolve_redirects"], "gold_files": ["src/requests/sessions.py"], "anchors": ["requests.models.Response", "requests.sessions.Session.send"], "unmapped_hunks": 0}, {"id": "requests_7309", "repo": "requests", "base_commit": "7407309c8a8a73aa2f4337184025d440bbedab7a", "title": "Fix malformed value parsing for Content-Type", "gold_functions": ["requests.utils._parse_content_type_header"], "gold_files": ["src/requests/utils.py"], "anchors": ["requests.utils._parse_content_type_header"], "unmapped_hunks": 0}]')
print('written')

In [ ]:
# ADDED. Load tasks and run a quick smoke test (2 tasks per repo, all three localizers)
import sys, json, time
sys.path.insert(0, '/kaggle/working')
import importlib, localizer as L
importlib.reload(L)
TASKSET = json.load(open('/kaggle/working/task_set_40.json'))
PROBS = {}
for line in open(DATA_DIR / 'tasks.jsonl'):
    d = json.loads(line); PROBS[d['instance_id']] = d['problem_statement']
chat = L.make_chat(base, INFERENCE_API_KEY, SERVED, max_tokens=500)
smoke = [t for r in ('fastapi', 'rich', 'requests') for t in [x for x in TASKSET if x['repo'] == r][:2]]
for cond in ('A1', 'A1.5', 'A2'):
    t0 = time.time(); rows = L.run(smoke, cond, GRAPH_DIR, PROBS, chat, workers=4)
    print(L.summarize(rows, cond), f'{time.time()-t0:.0f}s')
    print('  sample raw output', repr(rows[0]['raw'])[:300], '| read', rows[0]['read'])

In [ ]:
# ADDED. Full run on all 40 tasks. Look at the smoke test first. Each condition saves its own file.
lines = []
for cond in ('A1', 'A1.5', 'A2'):
    t0 = time.time(); rows = L.run(TASKSET, cond, GRAPH_DIR, PROBS, chat, workers=8, log=f'/kaggle/working/loc_{cond}.json')
    lines.append(L.summarize(rows, cond) + f'  [{time.time()-t0:.0f}s]')
    for repo in ('fastapi', 'rich', 'requests'):
        lines.append('   ' + L.summarize([r for r in rows if r['repo'] == repo], repo))
    print(*lines[-4:], sep='\n')
open('/kaggle/working/summary.txt', 'w').write('\n'.join(lines))